<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/main/pose-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pose detection: video → landmarks → angles

Runs MediaPipe Pose on a workout video and turns every frame into the same features as the Kaggle data used in `form_analysis.ipynb`: 33 landmarks, scaled the same way, plus the 12 joint angles (computed by the shared `features.py`).

Steps: set up → pick a video → find the pose in each frame → convert to the dataset's format → plot angles over time → save a copy of the video with the skeleton drawn on.

Film from the side or at an angle, with the whole body in frame.

In [ ]:
# Setup: MediaPipe, its pose model, and our shared features.py
!pip install -q mediapipe   # also installs OpenCV
!wget -q -nc https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!wget -q -nc https://raw.githubusercontent.com/Ekimxii/fitness_pose_tracker/main/features.py

In [ ]:
# Pick a video: in Colab a file picker opens; elsewhere, set VIDEO to a file path
VIDEO = "my_squats.mp4"
try:
    from google.colab import files
    VIDEO = next(iter(files.upload()))
except ImportError:
    pass

In [ ]:
# Find the pose in every frame
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks.python import BaseOptions, vision

def extract_pose(video_path, model="pose_landmarker_full.task"):
    """Run MediaPipe on every frame of a video.

    Returns the frame numbers where a person was found, their landmarks as an (n, 33, 3) array
    (x and y as fractions of the image width and height, z on the same scale as x),
    each landmark's visibility (0-1), the frame rate, and the frame size in pixels.
    """
    options = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=model),
                                           running_mode=vision.RunningMode.VIDEO)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames, points, visibility, size = [], [], [], None
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            size = (frame.shape[1], frame.shape[0])   # width, height
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = landmarker.detect_for_video(image, int(i * 1000 / fps))
            if result.pose_landmarks:   # frames where nobody is found are skipped
                frames.append(i)
                points.append([[lm.x, lm.y, lm.z] for lm in result.pose_landmarks[0]])
                visibility.append([lm.visibility for lm in result.pose_landmarks[0]])
            i += 1
    cap.release()
    assert frames, f"No person found in {video_path} ({i} frames read)"
    print(f"Person found in {len(frames)} of {i} frames; {size[0]}x{size[1]} at {fps:.0f} fps")
    return np.array(frames), np.array(points), np.array(visibility), fps, size

frames, points, visibility, fps, (width, height) = extract_pose(VIDEO)

In [ ]:
# Convert to the Kaggle dataset's format: same columns, same scaling, same 12 angles
from features import ALL_ANGLES, LANDMARK_COLS, LANDMARKS, add_angles, normalize

pixels = points * np.array([width, height, width])   # the dataset was built from pixel coordinates
poses = pd.DataFrame(normalize(pixels).reshape(len(frames), -1), columns=LANDMARK_COLS)
poses.insert(0, "frame", frames)
poses.insert(1, "time_s", frames / fps)
poses = add_angles(poses)

# How sure MediaPipe is that each joint is in view (0-1). Below about 0.5 the position is a guess,
# so don't judge form on those joints in those frames.
vis = pd.DataFrame(visibility, columns=LANDMARKS)
print("joints that are often hidden:", list(vis.columns[(vis < 0.5).mean() > 0.2]))
poses[list(ALL_ANGLES)].rename(columns=ALL_ANGLES).describe().round(0)

In [ ]:
# Angles over time: each rep should show up as one dip or peak
import matplotlib.pyplot as plt

SHOW = ["R knee", "R hip bend", "R elbow", "torso lean"]   # pick the joints that matter for the exercise
col = {label: name for name, label in ALL_ANGLES.items()}

fig, ax = plt.subplots(figsize=(10, 4))
for label in SHOW:
    smooth = poses[col[label]].rolling(5, center=True, min_periods=1).median()   # removes jitter
    ax.plot(poses["time_s"], smooth, label=label, lw=1.8)
ax.set(xlabel="time (s)", ylabel="angle (degrees)", ylim=(0, 185), title="Joint angles over time")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()

In [ ]:
# Save a copy of the video with the skeleton and live angles drawn on
OUT = "pose_overlay.mp4"
CONNECTIONS = [(c.start, c.end) for c in vision.PoseLandmarksConnections.POSE_LANDMARKS]
row_of = {f: r for r, f in enumerate(frames)}   # frame number -> row in poses

def put_text(img, text, org):
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4, cv2.LINE_AA)        # outline
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

cap = cv2.VideoCapture(VIDEO)
writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i in row_of:
        r = row_of[i]
        xy = (points[r, :, :2] * [width, height]).astype(int)
        seen = visibility[r] > 0.5
        for a, b in CONNECTIONS:
            if seen[a] and seen[b]:
                cv2.line(frame, tuple(map(int, xy[a])), tuple(map(int, xy[b])), (255, 255, 255), 2, cv2.LINE_AA)
        for p in xy[seen]:
            cv2.circle(frame, tuple(map(int, p)), 4, (214, 120, 42), -1, cv2.LINE_AA)   # BGR blue
        for k, label in enumerate(SHOW):
            put_text(frame, f"{label}: {poses[col[label]].iloc[r]:.0f} deg", (10, 30 + 28 * k))
    writer.write(frame)
    i += 1
cap.release()
writer.release()
print("saved", OUT)

try:
    files.download(OUT)   # Colab: save it to your computer
except NameError:
    pass